# Manual Tuning — the settings a model cannot learn

## Two kinds of number

When you call `.fit()`, a model learns numbers from the data: the coefficients of a linear
regression, the split thresholds of a tree. Those are **parameters**, and you never choose
them yourself.

But before `.fit()` runs, you have already made choices it cannot revisit. How strong should
the regularisation be? How deep may the tree grow? How many neighbours vote? These are
**hyperparameters** — settings that shape *how* learning happens, and that the fitting
process takes as given.

You have already met several without the label:

| Lesson | Hyperparameter | What it controlled |
|---|---|---|
| Bias-Variance | `alpha` in `Ridge` / `Lasso` | how hard coefficients are penalised |
| Logistic Regression | `C` | the same thing, inverted |
| Distance Metrics + KNN | `n_neighbors` | how many neighbours vote |
| Decision Trees | `max_depth`, `min_samples_leaf` | how far the tree may grow |

In every one of those lessons you chose a value by hand, or accepted the default. This lesson
is about doing it systematically.

In [ ]:
import numpy as np
import pandas as pd

from sklearn.compose import make_column_selector, make_column_transformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score, train_test_split
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

RSEED = 42

# Predict a penguin's sex from its measurements. Species prediction would be too easy —
# nearly perfectly separable — and a tuning lesson needs a task where the settings matter.
penguins = pd.read_csv("data/penguins.csv").dropna(subset=["sex"])

X = penguins.drop(columns="sex")
y = penguins["sex"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=RSEED, stratify=y
)

print(f"{len(X_train)} training penguins, {len(X_test)} held out")
print(y.value_counts().to_dict())

## The pipeline we will tune

Straight from the Feature Engineering lesson — impute, scale, encode, classify. Nothing new;
what is new is treating its settings as things to search over.

In [ ]:
numeric = make_pipeline(SimpleImputer(strategy="median"), StandardScaler())
categorical = make_pipeline(SimpleImputer(strategy="most_frequent"),
                            OneHotEncoder(handle_unknown="ignore"))

preprocessor = make_column_transformer(
    (numeric, make_column_selector(dtype_include="number")),
    (categorical, make_column_selector(dtype_exclude="number")),
)

model = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", LogisticRegression(max_iter=1000)),
])
model

## `get_params()` — what is there to tune?

You cannot tune what you cannot name. Every estimator exposes its settings through
`get_params()`, and for a pipeline the names are built by joining step names with a **double
underscore**.

In [ ]:
params = model.get_params()
print(f"{len(params)} parameters in total. The tunable-looking ones:\n")

for name in sorted(params):
    if name.count("__") >= 1 and not name.endswith(("steps", "transformers", "memory", "verbose")):
        print(f"  {name:62s} = {params[name]}")

That is the whole surface of the model, laid out.

Read one of the long names slowly:

```
preprocessor__pipeline-1__simpleimputer__strategy
     │              │            │            └── the parameter itself
     │              │            └── the step inside that inner pipeline
     │              └── the branch of the ColumnTransformer
     └── the step in the outer pipeline
```

**This is why the Feature Engineering lesson insisted on naming steps explicitly.** Our outer
steps are `preprocessor` and `classifier` because we named them. The inner ones are
`pipeline-1`, `simpleimputer` and so on because `make_pipeline` generated them — and those
are the names you now have to type.

## `set_params()` — changing one without rebuilding

You can reach any of those names and change it in place.

In [ ]:
print("before:", model.get_params()["classifier__C"])

model.set_params(classifier__C=0.01)
print("after :", model.get_params()["classifier__C"])

model.set_params(classifier__C=1.0)   # put it back

## Tuning by hand

Now the actual job. Try some values of `C`, score each with cross-validation, keep the best.

Note what we are **not** doing: touching the test set. Every score below comes from
cross-validation on the training data only.

In [ ]:
for C in [0.001, 0.01, 0.1, 1, 10, 100]:
    model.set_params(classifier__C=C)
    scores = cross_val_score(model, X_train, y_train, cv=5, scoring="accuracy")
    print(f"C = {C:<8} cv accuracy = {scores.mean():.4f} ± {scores.std():.4f}")

One line per candidate, and the winner is readable off the page.

### Two hyperparameters

Now suppose the imputation strategy also matters. Two settings means a nested loop, and the
output is a grid rather than a list.

### A gotcha worth meeting now

`make_pipeline` generated the name `pipeline-1`, **with a hyphen**. A hyphen is not legal in a
Python identifier, so this does not parse:

```python
model.set_params(preprocessor__pipeline-1__simpleimputer__strategy="median")   # SyntaxError
```

You have to unpack a dictionary instead:

```python
model.set_params(**{"preprocessor__pipeline-1__simpleimputer__strategy": "median"})
```

This is the concrete cost of the shorthand constructors, and the reason the Feature
Engineering lesson said to name steps explicitly for anything you intend to tune. Had we
written `ColumnTransformer([("num", numeric, …), ("cat", categorical, …)])`, the name would be
`preprocessor__num__simpleimputer__strategy` — legal as a keyword argument, and readable.

In [ ]:
rows = []
for C in [0.01, 0.1, 1, 10, 100]:
    for strategy in ["mean", "median"]:
        model.set_params(**{
            "classifier__C": C,
            "preprocessor__pipeline-1__simpleimputer__strategy": strategy,
        })
        score = cross_val_score(model, X_train, y_train, cv=5, scoring="accuracy").mean()
        rows.append({"C": C, "imputer": strategy, "cv accuracy": round(score, 4)})

pd.DataFrame(rows).pivot(index="C", columns="imputer", values="cv accuracy")

Ten fits, written as six lines of loop. Add a third hyperparameter with five values and it
is fifty fits and three levels of indentation; add a fourth and you are writing a small
program whose only job is bookkeeping.

**That is the motivation for the rest of this lesson.** `GridSearchCV` is this nested loop,
written once, by somebody else.

## The part people get wrong: you now need three splits

Here is the trap, and it is worth slowing down for.

We chose `C` by looking at cross-validated scores. The best of those scores was, say, 0.99.
Is 0.99 our estimate of how the model will perform?

**No.** We picked the winner *because* it scored highest. Across many candidates, some of
that height is real and some is luck — and by selecting the maximum we have kept the luck.
The winning score is biased upward, and the more candidates we try, the worse the bias.

The fix is to keep a portion of the data that plays no part in the choosing:

| Split | Used for | Seen how often |
|---|---|---|
| **train** | fitting each candidate | many times |
| **validation** | comparing candidates and picking one | many times |
| **test** | estimating how the chosen model performs | **exactly once, at the end** |

Cross-validation on the training set *is* the validation step — it carves the validation
portion out repeatedly rather than fixing one slice. The test set stays sealed.

In [ ]:
best_C = 1.0
model.set_params(**{"classifier__C": best_C,
                    "preprocessor__pipeline-1__simpleimputer__strategy": "median"})

cv_estimate = cross_val_score(model, X_train, y_train, cv=5).mean()

model.fit(X_train, y_train)
test_estimate = model.score(X_test, y_test)

print(f"best cross-validated score (used for choosing) : {cv_estimate:.4f}")
print(f"held-out test score        (the honest number) : {test_estimate:.4f}")

On this dataset the two land close together (cross-validated 0.924, held-out 0.929) because
we compared only ten candidates. **Do not read that as reassurance.** The gap grows with the
number of things you try, and a serious search tries hundreds.

## Summary

- **Parameters** are learned from data. **Hyperparameters** are chosen before fitting.
- `get_params()` lists every tunable name; the `step__parameter` convention addresses them
  through a pipeline; `set_params()` changes them in place.
- Tuning by hand is a nested loop, and it stops scaling almost immediately.
- The score you used to *choose* is not an estimate of performance. Keep a test set that the
  selection never touched.

> **Next:** `GridSearchCV` automates the loop — and handles the train/validation split
> internally so you cannot forget it.